# 21.19. MP-07 — прогноз движения сердца по одной 3D КТ

**Статус: выполнен реальный неподвижный контроль, прогноз движущейся моделью не выполнен.** Постановка и ограничения заданы в [21.14](21.14_Протокол_прогноза_сердца_по_одной_КТ.ipynb). Одна КТ означает трёхмерный том одной фазы, а не один срез. Проверяемый вопрос — улучшит ли будущая модель по одной опорной фазе геометрию сердца на остальных *действительно снятых* фазах по сравнению с неподвижным сердцем.


## Ник как основной тест одной КТ

Основной проверочный человек — Ник: входная фаза 00 и 11 скрытых проверочных фаз. Уже вычисленный неподвижный контроль даёт средний Dice `0,9177`, среднюю ошибку центра `1,894` мм, минимальный Dice `0,8521` на фазе 03 и максимальную разницу объёма `54,404` мл. Это сравнение автоматических масок с неподвижной формой, **не точность прогноза MONAI Physio**. Данные Adam и Georg ниже сохранены как дополнительные сравнения. Настоящая модель должна получать только одну КТ Ника, а обучение и настройка не должны видеть остальные его фазы.


## Входы, скрытие фаз и QC

**Факт о данных:** Adam — 12 фаз, Georg — 15, Nix — 12; для каждой есть 3D КТ и автоматическая `whole_heart` маска TotalSegmentator v2 в `products_v2`. Статус источника масок — `automatic_products_pending_manual_review`; независимая анатомическая точность не установлена. Опорная `phase_00` назначена заранее для каждого человека. У неё DICOM-метка номинальной доли R–R составляет 12%, 23% и 14% соответственно, а последняя фаза — 117%, 164% и 121%. Это метки сканирования, не доказанные моменты механического цикла. Последняя фаза не замыкается на первую. Состояние дыхания и неопределённость границ масок отдельно не установлены.

**Контроль без утечки:** предсказанием назначается неизменная маска `phase_00`; после её фиксации открываются маски фаз `01+` исключительно для оценки. Никакие параметры по скрытым фазам не подгоняются. Проверяются бинарность/непустота масок, соответствие сетки КТ и масок каждой фазы, совпадение аффинных матриц между фазами, ортогональность осей для физических поверхностных расстояний. Скрипт выводит SHA-256 масок и опорной КТ, не сохраняет массивы.


## Реальный контроль 28.09.2026

| Человек | Проверочных фаз | Средний Dice | Средняя ошибка центра, мм | Минимальный Dice | Максимальная ошибка объёма, мл |
|---|---:|---:|---:|---:|---:|
| Adam | 11 | 0,9299 | 1,090 | 0,8786 | 50,919 |
| Georg | 14 | 0,9250 | 1,453 | 0,8918 | 44,565 |
| Nix | 11 | 0,9177 | 1,894 | 0,8521 | 54,404 |

**Результат расчёта:** неподвижная форма даёт ненулевую ошибку уже на этих автоматических масках. У Nix минимальный Dice наблюдался на `phase_03`. Ошибка центра нулевой модели равна фактическому смещению центра маски от опорной фазы; предсказанное изменение объёма равно нулю, поэтому указанная ошибка объёма есть абсолютная величина изменения относительно опоры. Вычисляются также пофазные ASSD и HD95; полный вывод воспроизводится скриптом. Эти числа не являются измерением качества MeshGraphNet и не подтверждают физиологическое происхождение всей разности масок.


## Настоящий прогноз по одной КТ: что уже есть и что мешает

В [официальном старом выпуске](https://github.com/Project-MONAI/monai-physio/releases) обнаружены готовые сердечные веса. Они запущены на исходном среднем шаблоне в MP-06 без обучения на наших данных. Это ещё **не прогноз движения Ника**: сеть ждёт 15 параметров его формы на согласованном шаблоне, а для их определения нужна полная готовая PCA-модель и подгонка к КТ `phase_00`. Её большой файл пока не удалось получить целиком. Кроме того, автоматическая `whole_heart` маска не обязана иметь ту же анатомическую границу, что исходный Duke Heart шаблон.

Проверочный протокол остаётся прежним: использовать только КТ/маску фазы 00 для подгонки; заранее фиксировать стадии и прогноз; потом открыть остальные 11 реально снятых фаз Ника и сравнить положение, форму и объёмы с неподвижным контролем. Номинальные R–R проценты не доказывают точное механическое время. Пока такая проверка не завершена, числа неподвижного контроля не называются точностью нейросети.

## Решение и дальнейшая проверка

**Нерешённый вопрос:** готовых совместимых сердечных весов с проверенным происхождением пока нет, поэтому прогноз движения не получен; обучение на своих данных не планируется. Следовательно, улучшение над этим контролем и пригодность замены отсутствующей 4D КТ не доказаны. Порог допустимой геометрической и электрической ошибки должен быть задан до оценки будущего предсказателя. Для FEM нужны отдельные проверки сетки, материала, электродов, `Z` и `ΔZ`; Dice сам по себе их не заменяет.

Воспроизведение: [mp05_07_eval.py](mp05_07_eval.py), команда `python mp05_07_eval.py baseline --data-root <локальный_корень_производных_4D_КТ>` в среде с `numpy`, `scipy`, `nibabel`. Корень данных передаётся локально, абсолютный путь и первичные массивы не записываются в ноутбук или Git. Журнал входов и блокеров: [mp05_07_results.md](mp05_07_results.md); отдельный статус сети: [MP-06](21.18_MP06_Суррогат_движения.ipynb).
